# 🧠 Stochastic Gradient Descent (SGD)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Stochastic Gradient Descent (SGD)**! ในสมุดบันทึกนี้ เราจะ:
1. เปรียบเทียบ Batch Gradient Descent กับ Stochastic Gradient Descent ทั้งในเชิงแนวคิดและคณิตศาสตร์
2. อิมพลีเมนต์ทั้ง Batch GD และ SGD จากศูนย์โดยใช้ Python/NumPy
3. ติดตามและเปรียบเทียบเส้นทางการอัปเดตน้ำหนักบนแผนที่เส้นชั้นความสูงของลอสแบบ 2 มิติ (2D loss contour map)
4. พล็อตเส้นกราฟของลอสเพื่อดูภาพการลู่เข้าที่ราบรื่นของ Batch GD เทียบกับการแกว่งที่มีเสียงรบกวน (noisy fluctuations) ของ SGD
5. อธิบายบทบาทของสัญญาณรบกวนแบบสุ่ม (stochastic noise) ในการหลุดพ้นจากจุดต่ำสุดเฉพาะที่ (local minima) และข้อจำกัดด้านประสิทธิภาพของฮาร์ดแวร์ (นำไปสู่การใช้ Mini-Batch GD ใน YOLO)

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันเลย

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

## 1. การจำลองข้อมูลสำหรับการถดถอยเชิงเส้น (Linear Regression)

เราสร้างจุดข้อมูลจำนวน 100 จุดตามความสัมพันธ์เชิงเส้น:
$$y = 3x + 1 + \text{noise}$$

In [ ]:
n_samples = 100
X = np.random.rand(n_samples, 1)
y = 3 * X + 1 + np.random.normal(0, 0.15, (n_samples, 1))

plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='blue', edgecolor='k', s=40, label='Data Points')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Synthetic Linear Dataset')
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend()
plt.show()

## 2. การอิมพลีเมนต์ Batch GD และ SGD จากศูนย์ (from Scratch)

มาเขียนฟังก์ชันเพื่อติดตามเส้นทางของน้ำหนัก $w$ และอคติ $b$ (bias) ในขณะที่ทำการเพิ่มประสิทธิภาพ:
- ใน **Batch GD** เราทำการอัปเดต 1 ครั้งต่อรอบ (epoch) โดยใช้เกรเดียนต์เฉลี่ยจากข้อมูลตัวอย่างทั้งหมด 100 ตัว
- ใน **SGD** เราทำการอัปเดต 100 ครั้งต่อรอบ (epoch) (อัปเดตหลังจากตัวอย่างแต่ละตัว โดยสลับลำดับแบบสุ่ม)

In [ ]:
def run_batch_gd(X, y, lr=0.1, epochs=50):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        preds = w * X + b
        error = preds - y
        dw = (1 / m) * np.sum(error * X)
        db = (1 / m) * np.sum(error)
        w -= lr * dw
        b -= lr * db
        history.append([w, b])
        
    return np.array(history)

def run_sgd(X, y, lr=0.1, epochs=5):
    w, b = 0.0, 0.0
    m = len(X)
    history = [[w, b]]
    
    for _ in range(epochs):
        indices = np.random.permutation(m)
        for idx in indices:
            xi = X[idx]
            yi = y[idx]
            pred = w * xi + b
            error = pred - yi
            dw = error * xi
            db = error
            w -= lr * dw
            b -= lr * db
            history.append([float(w), float(b)])
            
    return np.array(history)

path_bgd = run_batch_gd(X, y, lr=0.1, epochs=50)
path_sgd = run_sgd(X, y, lr=0.02, epochs=5)

## 3. การแสดงเส้นทางการลู่เข้าบนแผนที่เส้นชั้นความสูงของลอสแบบ 2 มิติ (2D Loss Contour Map)

ลองคำนวณพื้นที่ผิวของลอส (Loss Landscape) ทั้งหมดแล้วพล็อตเส้นทางการเพิ่มประสิทธิภาพกัน

In [ ]:
# Compute loss grid
w_vals = np.linspace(-0.5, 4.5, 100)
b_vals = np.linspace(-0.5, 2.5, 100)
W, B = np.meshgrid(w_vals, b_vals)

Z = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        w_tmp, b_tmp = W[i, j], B[i, j]
        Z[i, j] = (1 / (2 * n_samples)) * np.sum((w_tmp * X + b_tmp - y) ** 2)

# Plot contours and trajectories
plt.figure(figsize=(10, 8))
contours = plt.contour(W, B, Z, levels=25, cmap='viridis')
plt.clabel(contours, inline=1, fontsize=8)

plt.plot(path_bgd[:, 0], path_bgd[:, 1], color='red', marker='o', linewidth=2.5, label='Batch GD (Smooth path)')
plt.plot(path_sgd[:, 0], path_sgd[:, 1], color='orange', alpha=0.7, linewidth=1.5, label='SGD (Noisy/Jagged path)')

plt.scatter(3.0, 1.0, color='blue', s=120, marker='*', zorder=5, label='Target Minimum')
plt.xlabel('Weight (w)')
plt.ylabel('Bias (b)')
plt.title('Optimization Paths: Batch GD vs. Stochastic GD')
plt.legend()
plt.show()

ดูพล็อตสิ!
-   **Batch GD (สีแดง):** เคลื่อนที่ในแนวเส้นตรงอย่างสมบูรณ์แบบตั้งฉากลงไปตามแนวเกรเดียนต์ ลู่เข้าช้าแต่เป็นระเบียบเรียบร้อย
-   **SGD (สีส้ม):** ใช้เส้นทางคดเคี้ยวไปมา (zigzag) ที่ปั่นป่วนมาก สัญญาณรบกวนแบบสุ่มทำให้พารามิเตอร์กระดอนไปรอบๆ แต่โดยรวมแล้วมันลู่เข้าหาจุดต่ำสุดได้เร็วกว่ามากในแง่ของข้อมูลที่ใช้ในการประมวลผล!

## 💡 ความเชื่อมโยงกับ YOLO และประสิทธิภาพของฮาร์ดแวร์
ทำไม YOLO จึงใช้ **Mini-Batch GD** แทนที่จะเป็น SGD แท้ๆ (ขนาดแบตช์ = 1)?
1.  **การทำให้เป็นเวกเตอร์และการประมวลผลแบบขนาน (Vectorization & Parallelism):** ตัวประมวลผล (โดยเฉพาะ GPU) ได้รับการออกแบบมาเพื่อคูณเมทริกซ์ การคูณเมทริกซ์น้ำหนักกับกลุ่มภาพขนาด 16 หรือ 32 ภาพใช้เวลาแทบจะเท่ากันกับการคูณด้วยภาพเพียง 1 ภาพ เนื่องจากการประมวลผลแบบขนาน
2.  **ความเสถียรของเกรเดียนต์ (Gradient Stability):** SGD แท้ๆ มีสัญญาณรบกวนมากเกินไป ทำให้ไม่สามารถลู่เข้าสู่ค่าความแม่นยำสูงได้ การทำ Mini-batch จะเฉลี่ยเกรเดียนต์จากภาพ 16-64 ภาพ ซึ่งช่วยให้อัปเดตได้เสถียรขึ้นในขณะที่ยังรักษาความเร็วในการประมวลผลที่สูงไว้ได้